<a href="https://colab.research.google.com/github/Bhoomikamalik81/Probation-Project-26/blob/main/bhoomika_task_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Bengaluru_House_Data.csv to Bengaluru_House_Data.csv


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

loading data

In [6]:
df = pd.read_csv("Bengaluru_House_Data.csv")
print("Dataset shape:", df.shape)

Dataset shape: (13320, 9)


In [7]:
print("First 5 rows:")
print(df.head())

First 5 rows:
              area_type   availability                  location       size  \
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK   
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom   
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK   
3  Super built-up  Area  Ready To Move        Lingadheeranahalli      3 BHK   
4  Super built-up  Area  Ready To Move                  Kothanur      2 BHK   

   society total_sqft  bath  balcony   price  
0  Coomee        1056   2.0      1.0   39.07  
1  Theanmp       2600   5.0      3.0  120.00  
2      NaN       1440   2.0      3.0   62.00  
3  Soiewre       1521   3.0      1.0   95.00  
4      NaN       1200   2.0      1.0   51.00  


In [8]:
print("Columns:")
print(df.columns.tolist())

Columns:
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']


In [9]:
print("Data types:")
print(df.dtypes)

Data types:
area_type        object
availability     object
location         object
size             object
society          object
total_sqft       object
bath            float64
balcony         float64
price           float64
dtype: object


In [10]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [11]:
print("Duplicate rows:")
print(df.duplicated().sum())

Duplicate rows:
529


In [12]:
df = df.drop_duplicates()

In [13]:
print("Shape after removing duplicates:")
print(df.shape)

Shape after removing duplicates:
(12791, 9)


In [14]:
columns_to_strip = [
    "size",
    "availability",
    "location",
    "society",
    "total_sqft",
    "area_type"
]
for col in columns_to_strip:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

In [15]:
print("Area type values:")
print(df["area_type"].value_counts())

Area type values:
area_type
Super built-up  Area    8317
Built-up  Area          2398
Plot  Area              1989
Carpet  Area              87
Name: count, dtype: Int64


In [16]:
print("Availability values:")
print(df["availability"].value_counts())

Availability values:
availability
Ready To Move    10172
18-May             292
18-Dec             284
18-Apr             269
18-Aug             187
                 ...  
16-Oct               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: Int64


In [17]:
print("Size values:")
print(df["size"].value_counts().head(20))

Size values:
size
2 BHK         4931
3 BHK         4120
4 Bedroom      824
4 BHK          574
3 Bedroom      535
1 BHK          521
2 Bedroom      314
5 Bedroom      291
6 Bedroom      191
1 Bedroom      104
8 Bedroom       84
7 Bedroom       82
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
1 RK            13
10 Bedroom      12
9 BHK            8
8 BHK            5
Name: count, dtype: Int64


In [18]:
print("Location values:")
print(df["location"].value_counts().head(20))

Location values:
location
Whitefield                  524
Sarjapur  Road              379
Electronic City             289
Kanakpura Road              249
Thanisandra                 232
Yelahanka                   210
Marathahalli                169
Hebbal                      161
Raja Rajeshwari Nagar       159
Uttarahalli                 157
Hennur Road                 149
Bannerghatta Road           149
7th Phase JP Nagar          135
Electronic City Phase II    128
Rajaji Nagar                107
Haralur Road                102
Bellandur                    93
KR Puram                     91
Electronics City Phase 1     88
Hoodi                        88
Name: count, dtype: Int64


In [19]:
df = df.dropna(subset=["location"]).copy()

In [21]:
df["room_count"] = pd.to_numeric(df["size"].str.split().str[0], errors="coerce")

In [22]:
df["property_type"] = df["size"].str.split().str[1]

In [23]:
print(df["room_count"].isnull().sum())
print(df["property_type"].isnull().sum())
print(df["property_type"].value_counts(dropna=False))

16
16
property_type
BHK        10273
Bedroom     2488
<NA>          16
RK            13
Name: count, dtype: int64


In [24]:
def convert_sqft(x):
    try:
        x = str(x).strip()
        if "-" in x:
            values = x.split("-")
            low = float(values[0])
            high = float(values[1])
            return (low + high) / 2
        return float(x)
    except:
        return np.nan

In [25]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [26]:
df.dtypes

,0
area_type,string[python]
availability,string[python]
location,string[python]
size,string[python]
society,string[python]
total_sqft,float64
bath,float64
balcony,float64
price,float64
room_count,float64


In [27]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["room_count"] = pd.to_numeric(df["room_count"], errors="coerce")

In [28]:
df = df.dropna(subset=["total_sqft", "price", "room_count"])

In [29]:
print(df["total_sqft"].head())

0    1056.0
1    2600.0
2    1440.0
3    1521.0
4    1200.0
Name: total_sqft, dtype: float64


In [30]:
df = df[
    (df["total_sqft"] > 0) &
    (df["price"] > 0) &
    (df["room_count"] > 0)
].copy()

In [31]:
df["society"] = df.groupby("location")["society"].transform(
    lambda x: x.fillna(
        x.mode().iloc[0] if not x.mode().empty else "Unknown"
    )
)

In [32]:
df["society"] = df["society"].fillna("Unknown")

In [33]:
print("Missing society:", df["society"].isnull().sum())

Missing society: 0


In [34]:
if "bath" in df.columns:
    df["bath"] = pd.to_numeric(
        df["bath"],
        errors="coerce"
    )
    df.loc[
        (df["bath"] < 0) | (df["bath"] > 20),
        "bath"
    ] = np.nan
    df["bath"] = df["bath"].fillna(
        df["bath"].median()
    )
else:
    df["bath"] = 0

In [35]:
print(df["bath"].describe())

count    12728.000000
mean         2.699010
std          1.294223
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max         18.000000
Name: bath, dtype: float64
